In [1]:
#Importing libraries

import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
#CNN Architecture

class CNNModel(nn.Module):
    def __init__(self, num_classes=15):
        super(CNNModel, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Linear(128 * 28 * 28, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

In [3]:
#Initialize Model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNNModel(num_classes=15).to(device)
print(model)

CNNModel(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Linear(in_features=100352, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=15, bias=True)
  )
)


In [4]:
#Checking the accuracy on the test set
def evaluate(model, test_loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f" Accuracy: {accuracy:.2f}%")
    return accuracy

In [5]:
#Loss and Optimizer

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [6]:
#Training

def train(model, train_loader, val_loader, epochs=5):
    for epoch in range(epochs):

        model.train()
        train_loss = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {train_loss/len(train_loader)}, train_accuracy: {evaluate(model, train_loader):.4f}, Val Accuracy: {evaluate(model, val_loader):.4f}")

In [7]:
import sys
sys.path.append(r"E:\Laptop\Final_project")

In [8]:
#Data Loading

from src.data_loader import get_dataloaders

dataset_path = r"E:\Laptop\Final_project\Data\PlantVillage"

train_loader, val_loader, test_loader, class_names = get_dataloaders(dataset_path)

In [9]:
#Run Training

train(model, train_loader, val_loader, epochs=5)

 Accuracy: 79.56%
 Accuracy: 80.52%
Epoch 1, Loss: 1.1917420102167973, train_accuracy: 79.5584, Val Accuracy: 80.5170
 Accuracy: 86.20%
 Accuracy: 86.72%
Epoch 2, Loss: 0.6667764316420112, train_accuracy: 86.1969, Val Accuracy: 86.7205
 Accuracy: 88.16%
 Accuracy: 88.37%
Epoch 3, Loss: 0.5213967757546796, train_accuracy: 88.1628, Val Accuracy: 88.3683
 Accuracy: 92.20%
 Accuracy: 91.76%
Epoch 4, Loss: 0.42643713931330535, train_accuracy: 92.1985, Val Accuracy: 91.7609


KeyboardInterrupt: 

In [ ]:
# #Checking the accuracy on the test set
# def evaluate(model, test_loader):
#     model.eval()
#     correct = 0
#     total = 0

#     with torch.no_grad():
#         for images, labels in test_loader:
#             images, labels = images.to(device), labels.to(device)
#             outputs = model(images)
#             _, predicted = torch.max(outputs.data, 1)
#             total += labels.size(0)
#             correct += (predicted == labels).sum().item()

#     accuracy = 100 * correct / total
#     print(f"Test Accuracy: {accuracy:.2f}%")
#     return accuracy

In [ ]:
#Evaluate the model on the test set
evaluate(model, test_loader)

 Accuracy: 86.99%


86.98740716822732

In [ ]:
#SAVE THE MODEL
torch.save(model.state_dict(), "baseline_cnn_model.pth")